# Customer Churn Prediction System
**Course Project / Resume Portfolio Asset**

## 1. Project Overview & Objectives
Customer churn (also known as customer attrition) is one of the most critical metrics for subscription-based businesses. Retaining an existing customer is significantly cheaper than acquiring a new one. This project builds a complete, end-to-end Machine Learning pipeline to predict customer churn using the **IBM Telco Customer Churn** dataset.

### Objectives:
1. **Load and Clean Data**: Handle missing values, data type mismatches, and encode features.
2. **Exploratory Data Analysis (EDA)**: Uncover demographic, service, and account-related patterns driving churn.
3. **Predictive Modeling**: Compare Logistic Regression, Random Forest, and XGBoost with hyperparameter tuning.
4. **Pipeline Export**: Save a unified scikit-learn `Pipeline` (preprocessor + model) for production/web app usage.
5. **Extract Business Insights**: Offer actionable business strategies based on feature importances.

## 2. Setup and Data Loading

In [ ]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    roc_auc_score, confusion_matrix, roc_curve, classification_report
)
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from xgboost import XGBClassifier
import pickle

# Set styling
sns.set_theme(style="whitegrid")
plt.rcParams["figure.figsize"] = (10, 6)
plt.rcParams["font.size"] = 12

In [ ]:
# Load raw dataset
data_path = os.path.join("data", "raw", "Telco-Customer-Churn.csv")
df = pd.read_csv(data_path)
print(f"Dataset Dimension: {df.shape[0]} rows, {df.shape[1]} columns")
df.head()

In [ ]:
# Inspect data types and missing values
df.info()

## 3. Data Cleaning & Missing Values

Notice that `TotalCharges` is read as `object` (string) instead of a numeric type. Let's find out why by inspecting empty/whitespace strings.

In [ ]:
# Find rows with empty spaces in TotalCharges
empty_charges = df[df["TotalCharges"].str.strip() == ""]
print(f"Number of rows with empty TotalCharges: {len(empty_charges)}")
empty_charges[["tenure", "MonthlyCharges", "TotalCharges"]].head()

**Observation**: The rows with empty `TotalCharges` correspond to new customers with a `tenure` of `0`. Since they have just signed up, they haven't been billed yet. We should handle this by setting `TotalCharges` to `0.0` for these rows, and converting the column to float.

In [ ]:
# Replace empty spaces with NaN, convert to float, then fill NaN with 0
df["TotalCharges"] = df["TotalCharges"].replace(r'^\s*$', np.nan, regex=True)
df["TotalCharges"] = pd.to_numeric(df["TotalCharges"])
df["TotalCharges"] = df["TotalCharges"].fillna(0.0)

# Map Churn target to binary (Yes: 1, No: 0)
df["Churn"] = df["Churn"].map({"Yes": 1, "No": 0})

# Drop customerID (non-predictive unique ID)
df_clean = df.drop(columns=["customerID"])

# Verify types again
df_clean.info()

## 4. Exploratory Data Analysis (EDA)

Let's visualize relationships between key features and churn to uncover valuable insights.

In [ ]:
# Plot Churn distribution
plt.figure(figsize=(6, 4))
sns.countplot(x="Churn", data=df_clean, palette="Blues_r")
plt.title("Distribution of Customer Churn")
plt.xticks([0, 1], ["No Churn", "Churn"])
plt.ylabel("Count")
plt.show()

churn_rate = df_clean["Churn"].mean() * 100
print(f"Overall Churn Rate: {churn_rate:.2f}%")

In [ ]:
# 1. Tenure vs Churn
plt.figure(figsize=(10, 5))
sns.histplot(data=df_clean, x="tenure", hue="Churn", kde=True, multiple="stack", palette="crest", bins=30)
plt.title("Customer Churn by Tenure (Months)")
plt.xlabel("Tenure (Months)")
plt.ylabel("Count")
plt.show()

**Insight**: Customers with short tenure (0 to 12 months) are at a much higher risk of churning. Retention efforts should focus heavily on onboarding and the first year.

In [ ]:
# 2. Contract Type vs Churn
plt.figure(figsize=(8, 5))
sns.barplot(x="Contract", y="Churn", data=df_clean, palette="Set2", ci=None)
plt.title("Churn Rate by Contract Type")
plt.ylabel("Churn Rate (Probability)")
plt.ylim(0, 1)
plt.show()

**Insight**: Month-to-month customers have a churn rate exceeding 40%, while one-year and two-year contract customers are highly stable (less than 15% and 5% churn respectively).

In [ ]:
# 3. Online Security vs Churn
plt.figure(figsize=(8, 5))
sns.barplot(x="OnlineSecurity", y="Churn", data=df_clean, palette="viridis", ci=None)
plt.title("Churn Rate by Online Security Service")
plt.ylabel("Churn Rate (Probability)")
plt.ylim(0, 1)
plt.show()

**Insight**: Customers who do *not* have Online Security are significantly more likely to churn (over 30%) than those who do (under 15%).

In [ ]:
# 4. Payment Method vs Churn
plt.figure(figsize=(10, 5))
sns.barplot(x="PaymentMethod", y="Churn", data=df_clean, palette="muted", ci=None)
plt.title("Churn Rate by Payment Method")
plt.ylabel("Churn Rate (Probability)")
plt.xticks(rotation=15)
plt.ylim(0, 1)
plt.show()

**Insight**: Customers paying via **Electronic Check** churn at a much higher rate (nearly 45%) compared to other payment options (automatic bank transfers, credit cards, or mailed checks, which are around 15-20%).

## 5. Feature Engineering & Train-Test Split

Let's partition the clean dataset and set up our preprocessing module.

In [ ]:
numerical_cols = ["tenure", "MonthlyCharges", "TotalCharges"]
categorical_cols = [
    "gender", "SeniorCitizen", "Partner", "Dependents", "PhoneService",
    "MultipleLines", "InternetService", "OnlineSecurity", "OnlineBackup",
    "DeviceProtection", "TechSupport", "StreamingTV", "StreamingMovies",
    "Contract", "PaperlessBilling", "PaymentMethod"
]

X = df_clean.drop(columns=["Churn"])
y = df_clean["Churn"]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

print(f"Training set shape: {X_train.shape}")
print(f"Testing set shape: {X_test.shape}")

In [ ]:
# Define column transformer for Scaling & Encoding
preprocessor = ColumnTransformer(
    transformers=[
        ("num", StandardScaler(), numerical_cols),
        ("cat", OneHotEncoder(handle_unknown="ignore", sparse_output=False), categorical_cols)
    ]
)

## 6. Model Training and Hyperparameter Tuning

We fit the preprocessor on training data to feed GridSearchCV, optimizing F1-Score for three models.

In [ ]:
X_train_pre = preprocessor.fit_transform(X_train)
X_test_pre = preprocessor.transform(X_test)

# Setup tuning parameters
lr_grid = GridSearchCV(
    LogisticRegression(max_iter=1000, random_state=42),
    param_grid={"C": [0.01, 0.1, 1.0, 10.0], "solver": ["liblinear", "lbfgs"]},
    cv=5, scoring="f1", n_jobs=-1
)

rf_grid = GridSearchCV(
    RandomForestClassifier(random_state=42),
    param_grid={"n_estimators": [100, 200], "max_depth": [5, 10, None], "min_samples_split": [2, 5]},
    cv=5, scoring="f1", n_jobs=-1
)

xgb_grid = GridSearchCV(
    XGBClassifier(random_state=42, eval_metric="logloss"),
    param_grid={"n_estimators": [100, 200], "max_depth": [3, 5, 7], "learning_rate": [0.01, 0.1, 0.2]},
    cv=5, scoring="f1", n_jobs=-1
)

print("Tuning Logistic Regression...")
lr_grid.fit(X_train_pre, y_train)

print("Tuning Random Forest...")
rf_grid.fit(X_train_pre, y_train)

print("Tuning XGBoost...")
xgb_grid.fit(X_train_pre, y_train)

print("Model tuning completed successfully!")

## 7. Model Evaluation and Comparison

Let's evaluate the optimized classifiers on the test set.

In [ ]:
models = {
    "Logistic Regression": lr_grid.best_estimator_,
    "Random Forest": rf_grid.best_estimator_,
    "XGBoost": xgb_grid.best_estimator_
}

metrics = []
plt.figure(figsize=(10, 8))

for name, clf in models.items():
    y_pred = clf.predict(X_test_pre)
    y_prob = clf.predict_proba(X_test_pre)[:, 1]
    
    acc = accuracy_score(y_test, y_pred)
    prec = precision_score(y_test, y_pred)
    rec = recall_score(y_test, y_pred)
    f1 = f1_score(y_test, y_pred)
    auc = roc_auc_score(y_test, y_prob)
    
    metrics.append({
        "Model": name,
        "Accuracy": acc,
        "Precision": prec,
        "Recall": rec,
        "F1-Score": f1,
        "ROC-AUC": auc
    })
    
    fpr, tpr, _ = roc_curve(y_test, y_prob)
    plt.plot(fpr, tpr, label=f"{name} (AUC = {auc:.3f})")

plt.plot([0, 1], [0, 1], 'k--', label="Random Guess")
plt.xlabel("False Positive Rate")
plt.ylabel("True Positive Rate")
plt.title("ROC Curves comparison on Test Set")
plt.legend()
plt.show()

metrics_df = pd.DataFrame(metrics)
metrics_df

### Confusion Matrix (Logistic Regression)

Let's view the confusion matrix for our primary selected model (Logistic Regression).

In [ ]:
best_clf = lr_grid.best_estimator_
y_pred = best_clf.predict(X_test_pre)
cm = confusion_matrix(y_test, y_pred)

plt.figure(figsize=(6, 5))
sns.heatmap(cm, annot=True, fmt="d", cmap="Blues", cbar=False,
            xticklabels=["No Churn", "Churn"], yticklabels=["No Churn", "Churn"])
plt.ylabel("Actual Label")
plt.xlabel("Predicted Label")
plt.title("Confusion Matrix - Logistic Regression (Best F1)")
plt.show()

## 8. Exporting Unified Pipeline

We now instantiate a unified pipeline using Scikit-Learn `Pipeline` which contains the fitted preprocessing and the classification algorithm. This is pickled for downstream tasks.

In [ ]:
# Create unified pipeline
churn_pipeline = Pipeline(steps=[
    ("preprocessor", preprocessor),
    ("classifier", best_clf)
])

# Refit on entire raw dataset split to encapsulate fitted scaling
churn_pipeline.fit(X_train, y_train)

# Save
os.makedirs("models", exist_ok=True)
with open(os.path.join("models", "churn_pipeline.pkl"), "wb") as f:
    pickle.dump(churn_pipeline, f)
print("Unified pipeline saved successfully to models/churn_pipeline.pkl")

## 9. Business Insights & Actionable Recommendations

An analysis of feature coefficients from Logistic Regression reveals the core drivers of churn:

### Core Findings:
1. **Contract Type**: Month-to-month contracts are the strongest predictor of churn. Offering discounts to convert them to 1 or 2-year contracts can drastically drop attrition rates.
2. **Internet Service (Fiber Optic)**: Interestingly, customers with Fiber Optic service are more likely to churn, indicating potential pricing complaints or technical setup issues. A product check or survey is recommended.
3. **Add-on Services (Online Security & Tech Support)**: Customers without Online Security and Tech Support have a much higher churn probability. Bundling these features for free during onboarding could drive retention.
4. **Payment Method (Electronic Check)**: Electronic Check users represent the highest churn risk. Moving them to Automatic Credit Card or Bank Transfer payments by providing a small incentive (e.g. ₹150 bill credit) would reduce passive churn.
5. **Onboarding Risk**: Retention programs should actively target customers during their first 12 months, as this is the period of highest risk.